In [ ]:
#Run imports for libraries to be used
import pandas as pd
import json
import numpy as np
import sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from scipy.stats import zscore
from statistics import linear_regression
from sklearn.preprocessing import PowerTransformer
from imblearn.over_sampling import RandomOverSampler
import re
from sklearn.model_selection import cross_validate
from sklearn.impute import SimpleImputer
import sklearn
from imblearn.over_sampling import SMOTE
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.tree import DecisionTreeRegressor
from imblearn.pipeline import Pipeline as ImbPipeline



#Set up display options for pandas return functions
pd.options.display.max_rows = 100
pd.options.display.max_columns = 110
sklearn.set_config(transform_output="pandas")

In [ ]:
#load the schema
with open('ingest_schema.json') as f:
    schema = json.load(f)

#Use the schema and other params to ingest the csv
df_all = pd.read_csv(
    filepath_or_buffer ="CW_data.csv",
    encoding='ANSI',
    dtype = schema,
    true_values = ["positive", "detected"],
    false_values = ["negative", "not_detected"],
    on_bad_lines = "warn"
    )
#clean column labels for invisible space characters, create function for use on unseen data.
def clean_cols(df):
    # Remove occurrences of \xa0
    df.columns = [re.sub(r'\s+', ' ', col).strip() for col in df.columns]
    return df
df_all = clean_cols(df_all)
#drop bad cols & Rows (ID, NaN cols)
df_all.drop('Patient ID', axis = 1, inplace = True)
df_single_values =  df_all.loc[:, df_all.nunique(dropna = False) == 1]
df_all = df_all.drop(labels = df_single_values.columns, axis = 1)
ignored_cols = ["Patient age quantile", "SARS-Cov-2 exam result"]
all_feats_cols = df_all.columns.difference(ignored_cols)
df_all = df_all.dropna(how = "all", subset = all_feats_cols)
print(f'The shape of df_all: {df_all.shape}')



def prepare_data(nan_threshold, over_sample_type, over_sample_method, pca_components,imputer_type , imputer_strat, input_df, pca_skip):
    df_temp = input_df.copy()
    # Threshold values for NaN Cols
    missing_per_column = df_temp.isnull().sum(axis=0)
    threshold = missing_per_column < nan_threshold
    df_thresholded = df_temp.loc[:,threshold]
    # Split Classifiers
    df_classifiers = df_thresholded['SARS-Cov-2 exam result']
    df_predictors = df_thresholded.drop('SARS-Cov-2 exam result', axis = 1).copy()
    print(f'predictors post nan drops is: {df_predictors.shape}')
    #splitting test and train sets
    predictor_train, predictor_test, classifier_train, classifier_test = train_test_split(df_predictors,df_classifiers, test_size = 0.2, random_state = 42, stratify = df_classifiers)
    # imputing
    if imputer_type == 'simple':
        print(f'performing simple imputation using {imputer_strat}')
        imputer = SimpleImputer(strategy = imputer_strat)
        imputer.set_output(transform = "pandas")
        predictor_train = imputer.fit_transform(predictor_train)
        predictor_test = imputer.transform(predictor_test)
    elif imputer_type == 'mice':
        print('performing MICE imputation')
        imputer = IterativeImputer(max_iter = 100, random_state = 42, initial_strategy=imputer_strat, tol = 1e-2)
        imputer.set_output(transform = "pandas")
        predictor_train = imputer.fit_transform(predictor_train)
        predictor_test = imputer.transform(predictor_test)
    elif imputer_type == 'tree':
            print('performing tree regression imputation')
            imputer = IterativeImputer(
                estimator = DecisionTreeRegressor(max_features = 'sqrt', random_state = 42),
                random_state = 42,
                max_iter = 50,
                initial_strategy = imputer_strat)
            imputer.set_output(transform = "pandas")
            imputer.fit(predictor_train)
            predictor_train = imputer.fit_transform(predictor_train)
            predictor_test = imputer.transform(predictor_test)
    else:
        print('imputer type must be simple, tree or mice')
    # Skewing
    print('Performing Yeo-Johnson transformation')
    cols_for_transform = df_thresholded.select_dtypes(exclude = ['boolean']).columns
    transformer = PowerTransformer(method = 'yeo-johnson', standardize = True)
    transformer.set_output(transform="pandas")
    transformed_data = transformer.fit_transform(predictor_train[cols_for_transform].astype(float))
    transformed_data_test = transformer.transform(predictor_test[cols_for_transform].astype(float))
    predictor_train[cols_for_transform] = transformed_data
    predictor_test[cols_for_transform] = transformed_data_test
    # PCA
    if not pca_skip :
        print(f'Perfoming PCA with {pca_components} components')
        pca = PCA(n_components = pca_components)
        pca.set_output(transform = "pandas")
        predictor_train = pca.fit_transform(predictor_train)
        predictor_test = pca.transform(predictor_test)
    # Oversampling
    if over_sample_type == 'random':
        ros = RandomOverSampler(random_state = 42, sampling_strategy = over_sample_method)
        predictor_train, classifier_train = ros.fit_resample(predictor_train, classifier_train)
        return predictor_train, classifier_train, predictor_test, classifier_test
    #SMOTE oversampling
    elif over_sample_type == 'smote':
        sos = SMOTE(random_state = 42, sampling_strategy = over_sample_method)
        predictor_train, classifier_train = sos.fit_resample(predictor_train, classifier_train)
        return predictor_train, classifier_train, predictor_test, classifier_test
    else:
        return predictor_train, classifier_train, predictor_test, classifier_test

def run_models(predictor_train, classifier_train, predictor_test,random_state, rf_trees, rf_criterion, neighbors):
    rfclf = RandomForestClassifier(class_weight = 'balanced',random_state = random_state, n_estimators = rf_trees, criterion = rf_criterion, n_jobs = -1)
    rfclf.fit(predictor_train, classifier_train)
    predictions_rfclf = rfclf.predict(predictor_test)
    knn = KNeighborsClassifier(n_neighbors = neighbors)
    knn.fit(predictor_train, classifier_train)
    predictions_knn = knn.predict(predictor_test)
    return predictions_rfclf,predictions_knn, rfclf, knn

def cross_val(model, X, y, skf):
    results_cv = cross_validate(model, X, y, cv = skf, scoring = ['f1'], return_train_score = False)
    model_name = model.__class__.__name__
    score = f'F1 cross val score Mean for {model_name} model was: {results_cv['test_f1'].mean()}'
    return score

def create_pipeline(model):
    return ImbPipeline([
        ('imputer' , IterativeImputer(
                estimator = DecisionTreeRegressor(max_features = 'sqrt', random_state = 42),
                random_state = 42,
                max_iter = 50,
                initial_strategy = 'median'
        )),
        ('yeo-johnson' , PowerTransformer(method = 'yeo-johnson', standardize = True)),
        ('smote',SMOTE(random_state = 42, sampling_strategy = 'minority')),
        ('classifier', model)
    ])

# prepare data and pass the returned splits as vars
pred_train, class_train, pred_test, class_test = prepare_data(
    nan_threshold = 1100,
    over_sample_type = 'smote',
    over_sample_method = 'minority',
    pca_components = 5,
    imputer_type = 'mice',
    imputer_strat = 'median',
    input_df = df_all,
    pca_skip = True
    )

#Run models
predictions_rfclf, predictions_knn, rf_model, knn_model = run_models(
    pred_train,
    class_train,
    pred_test,
    random_state = 42,
    rf_trees = 200,
    rf_criterion = 'gini',
    neighbors = 5
    )

print(f'RF F1 is: {f1_score(class_test, predictions_rfclf)}\nKNN F1 is: {f1_score(class_test, predictions_knn)}')
# cross validate
# skf = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)
# print(f'Cross Val Scores are:\n{cross_val(model = regression_model, X = pred_test, y = class_test, skf = skf)} \n{cross_val(rf_model, pred_test, class_test, skf = skf)}\n{cross_val(knn_model, pred_test, class_test, skf = skf)}')

In [ ]:
cm_rfclf = confusion_matrix(class_test, predictions_rfclf)
plt.figure(figsize = (5,5))
sns.heatmap(cm_rfclf, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
plt.xlabel('Predicted Covid Result')
plt.ylabel('True Covid Result')
plt.show()

In [ ]:
cm_knn = confusion_matrix(class_test, predictions_knn)
plt.figure(figsize = (5,5))
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
plt.xlabel('Predicted Covid Result')
plt.ylabel('True Covid Result')
plt.show()

In [ ]:
#load the schema
with open('ingest_schema.json') as f:
    schema = json.load(f)

#Use the schema and other params to ingest the csv
df_all = pd.read_csv(
    filepath_or_buffer ="CW_data.csv",
    encoding='ANSI',
    dtype = schema,
    true_values = ["positive", "detected"],
    false_values = ["negative", "not_detected"],
    on_bad_lines = "warn"
    )
#clean column labels for invisible space characters, create function for use on unseen data.
def clean_cols(df):
    # Remove occurrences of \xa0
    df.columns = [re.sub(r'\s+', ' ', col).strip() for col in df.columns]
    return df
df_all = clean_cols(df_all)
#drop bad cols & Rows (ID, NaN cols)
df_all.drop('Patient ID', axis = 1, inplace = True)
df_single_values =  df_all.loc[:, df_all.nunique(dropna = False) == 1]
df_all = df_all.drop(labels = df_single_values.columns, axis = 1)
ignored_cols = ["Patient age quantile", "SARS-Cov-2 exam result"]
all_feats_cols = df_all.columns.difference(ignored_cols)
df_all = df_all.dropna(how = "all", subset = all_feats_cols)
print(f'The shape of df_all: {df_all.shape}')
missing_per_column = df_all.isnull().sum(axis=0)
threshold = missing_per_column < 1100
df_all = df_all.loc[:,threshold].copy()
# Split Classifiers
df_classifiers = df_all['SARS-Cov-2 exam result']
df_predictors = df_all.drop('SARS-Cov-2 exam result', axis = 1).copy()
# trying pipelines
models = [
    LogisticRegression(random_state = 42),
    RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'),
    KNeighborsClassifier(n_neighbors=5)
]
counts = df_all.dtypes.value_counts()
print(counts)
boolean_cols = df_all.select_dtypes(include = 'boolean').columns
df_all[boolean_cols] = df_all[boolean_cols].astype('Float64')
df_classifiers = df_all['SARS-Cov-2 exam result']
df_predictors = df_all.drop('SARS-Cov-2 exam result', axis = 1).copy()
skf = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)


for model in models:
    pipeline = create_pipeline(model)
    cv_scores = cross_val_score(pipeline, df_predictors, df_classifiers, cv = skf, scoring = 'f1')
    print(f"{model.__class__.__name__} F1 Mean: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")